# Multimodale Routenoptimierung

Dieses Notebook berechnet die optimal gewichtete Route (Kosten, Zeit, CO2) in einem multimodalen Transportnetzwerk.

## Zielsetzung
- Minimierung einer gewichteten Zielfunktion aus Kosten, Dauer und Emissionen.
- Einhaltung von Budget- und Zeitrestriktionen.

## Mathematische Grundlagen

### Mengen und Zustände

| Symbol | Bedeutung |
|---|---|
| $H$ | Hubs |
| $M$ | Modi: road, rail, air, ship |
| $M_h \subseteq M$ | am Hub $h$ verfügbare Modi (`supported_modes`) |
| $T=\{0,\ldots,T_{\max}\}$ | Zeitpunkte im Planungshorizont ($T_{\max}=48$ im Beispiel) |
| $L$ | Transportverbindungen mit Fahrplan (`arc_templates`) |
| $w$ | Gewicht der Sendung in Tonnen |
| $v_m$ | Geschwindigkeit des Modus $m$ |
| $c_m$ | Kostenfaktor des Modus $m$ in EUR pro Tonnenkilometer |
| $e_m$ | Emissionsfaktor des Modus $m$ in kg CO$_2$ pro Tonnenkilometer |

Ein Zeitknoten ist $n=(h,m,t)$ mit $h\in H$, $m\in M_h$ und $t\in T$.

### Kanten des zeitexpandierten Netzes

$$A=A_{\text{trans}}\cup A_{\text{wait}}\cup A_{\text{transfer}}.$$

| Kantenart | Bedeutung | Datenbasis |
|---|---|---|
| $A_{\text{trans}}$ | Fahrt zwischen Hubs | `arc_templates` |
| $A_{\text{wait}}$ | Warten in Hub und Modus | `supported_modes` |
| $A_{\text{transfer}}$ | Moduswechsel am Hub | `transfer_windows` |

#### Transportkanten

Eine Verbindung $l\in L$ wird als Tupel definiert:

$$l=(h_l^-,h_l^+,m_l,d_l,D_l^{\text{day}}).$$

| Bestandteil | Bedeutung | Bedingung |
|---|---|---|
| $h_l^-$ | Start-Hub | $h_l^-\in H$ |
| $h_l^+$ | Ziel-Hub | $h_l^+\in H$ |
| $m_l$ | Transportmodus | $m_l\in M$ |
| $d_l$ | Distanz in km | $d_l\geq 0$ |
| $D_l^{\text{day}}$ | tägliche Abfahrtsstunden | Stunden von 0 bis 23 (`departure_hours`) |

$\Delta t_l$ ist die Fahrzeit der Verbindung $l$ in Zeitschritten. Da ein Zeitschritt einer Stunde entspricht, wird die Fahrzeit auf die nächste volle Stunde aufgerundet:

$$\Delta t_l=\left\lceil\frac{d_l}{v_{m_l}}\right\rceil.$$


$P(T_{\max})$ bezeichnet die im Planungshorizont betrachteten Tage:

$$P(T_{\max})=\left\{0,\ldots,\left\lceil\frac{T_{\max}}{24}\right\rceil-1\right\}.$$

Für eine tägliche Abfahrtsstunde $\theta\in D_l^{\text{day}}$ gilt:

$$D_l=\{\theta+24p\mid \theta\in D_l^{\text{day}},\ p\in P(T_{\max})\}.$$

$$A_{\text{trans}}=
\left\{((h_l^-,m_l,t),(h_l^+,m_l,t+\Delta t_l))\ \middle|\ 
l\in L,\ m_l\in M_{h_l^-}\cap M_{h_l^+},\ t\in D_l,\ t+\Delta t_l\leq T_{\max}\right\}.$$


#### Wartekanten

$$A_{\text{wait}}=
\left\{((h,m,t),(h,m,t+1))\ \middle|\ h\in H,\ m\in M_h,\ t<T_{\max}\right\}.$$


#### Transferkanten

Eine Transferkante beschreibt das Umladen am selben Hub von einem bisherigen Modus $m_1$ auf einen neuen Modus $m_2$.

| Symbol | Bedeutung |
|---|---|
| $m_1$ | Modus vor dem Umladen |
| $m_2$ | Modus nach dem Umladen |
| $\Theta_{h,m_1,m_2}$ | tägliche Startstunden für diesen Wechsel am Hub $h$ |
| $\tau^{\text{tr}}=2$ | Dauer des Umladens in Stunden |

Über mehrere Planungstage ergeben sich die zulässigen Startzeitpunkte als

$$D^{\text{tr}}_{h,m_1,m_2}=\{\theta+24p\mid \theta\in\Theta_{h,m_1,m_2},\ p\in P(T_{\max})\}.$$

$$A_{\text{transfer}}=
\left\{((h,m_1,t),(h,m_2,t+\tau^{\text{tr}}))\ \middle|\ 
h\in H,\ m_1,m_2\in M_h,\ t\in D^{\text{tr}}_{h,m_1,m_2},\ t+\tau^{\text{tr}}\leq T_{\max}\right\}.$$

### Kosten und Emissionen

Jede Kante $a$ erhält einen Kostenwert $c(a)$ und einen Emissionswert $e(a)$. Bei einer **Transportkante** hängen beide Werte von drei Größen ab:

| Größe | Bedeutung |
|---|---|
| $d_l$ | Länge der gefahrenen Verbindung in km |
| $c_{m_l}$ | Transportkosten des verwendeten Modus je Tonnenkilometer; Einheit: EUR/(t km) |
| $e_{m_l}$ | Transportemissionen des verwendeten Modus je Tonnenkilometer; Einheit: kg CO$_2$/(t km) |
| $w$ | transportiertes Gewicht in Tonnen |

Berechnung:


| Größe | Kosten | Emissionen |
|---|---|---|
| Formel | $c(a)=d_l\cdot c_{m_l}\cdot w$ | $e(a)=d_l\cdot e_{m_l}\cdot w$ |




## Optimierungsmodell

### Parameter (Gewichte und Schranken)

* $w_c, w_t, w_e \in [0, 1]$: Gewichte für Kosten, Zeit und Emissionen ($\sum w = 1.0$)
* $C_{\max}, T_{\max}, E_{\max}$: Normalisierungsfaktoren (`MAX_EST_...`)
* $B$: Maximales Budget (`MAX_PRICE`)
* $h_{\text{start}}, h_{\text{end}}$: Start- und Ziel-Hub
* $t_{\text{start}}, t_{\text{deadline}}$: Startzeitpunkt und späteste Ankunftszeit

### Entscheidungsvariable

Für jede Kante $a \in A$ (wobei $A$ die Gesamtmenge aller generierten Arcs ist) definieren wir eine binäre Variable:

$$x_a = \begin{cases} 1, & \text{wenn Kante } a \text{ genutzt wird} \\ 0, & \text{sonst} \end{cases}$$

Jede Kante $a$ besitzt die Eigenschaften $c_a$ (Kosten), $d_a$ (Dauer/Time) und $e_a$ (Emissionen).

---

### Zielfunktion

Das Modell minimiert die gewichtete, normalisierte Summe aus Transportkosten, Transportdauer und CO₂-Emissionen entlang des gewählten Pfades:

$$\min_{x} \sum_{a \in A} x_a \cdot \left( w_c \cdot \frac{c_a}{C_{\max}} + w_t \cdot \frac{d_a}{T_{\max}} + w_e \cdot \frac{e_a}{E_{\max}} \right)$$

---

### Restriktionen

#### Budgetbeschränkung

Die Summe der Kosten aller gewählten Kanten darf das Budget $B$ nicht überschreiten:

$$\sum_{a \in A} x_a \cdot c_a \leq B$$

#### Flusserhaltung

Seien $N$ die Zeitknoten und $A\subseteq N\times N$ die gerichteten Kanten $a=(i,j)$.

$$A^{\mathrm{in}}(n)=\{(i,n)\in A\mid i\in N\},\qquad
A^{\mathrm{out}}(n)=\{(n,j)\in A\mid j\in N\}.$$

$$N_{\mathrm{start}}=\{(h,m,t)\in N\mid h=h_{\mathrm{start}},\ t=t_{\mathrm{start}}\},$$

$$N_{\mathrm{end}}=\{(h,m,t)\in N\mid h=h_{\mathrm{end}},\ t\leq t_{\mathrm{deadline}}\}.$$

$$\sum_{a\in A^{\mathrm{in}}(n)}x_a-\sum_{a\in A^{\mathrm{out}}(n)}x_a=0
\qquad \forall n\in N\setminus(N_{\mathrm{start}}\cup N_{\mathrm{end}}).$$

#### Startbedingung

$$\sum_{n \in N_{\text{start}}}\ \sum_{a \in A^{\text{out}}(n)} x_a=1.$$

#### Zielbedingung

$$\sum_{n \in N_{\text{end}}}\ \sum_{a \in A^{\text{in}}(n)} x_a=1.$$

## Implementierung

### Datenstruktur und Generierung des zeitexpandierten Netzes

Wir erstellen drei Arten von Kanten:
1. **Transport-Arcs**: Bewegung zwischen Hubs.
2. **Waiting-Arcs**: Verweilen an einem Hub (Lagerung).
3. **Transfer-Arcs**: Wechsel zwischen Verkehrsträgern an einem Hub.

In [17]:
import pulp
import numpy as np

# 1. Basisdaten
hubs = [
    {
        "id": "BER",
        "name": "Berlin",
        "supported_modes": {"road", "rail", "air", "ship"},
        "transfer_windows": {
            ("road", "rail"): [4, 5, 6, 16, 17],
            ("rail", "road"): [9, 10, 21, 22],
            ("road", "air"): [7, 8, 15, 16],
            ("air", "road"): [11, 12, 19, 20],
            ("road", "ship"): [2, 3, 14, 15],
            ("ship", "road"): [8, 9, 20, 21],
            ("rail", "ship"): [5, 17],
            ("ship", "rail"): [11, 23],
        },
    },
    {
        "id": "HAM",
        "name": "Hamburg",
        "supported_modes": {"road", "rail", "air", "ship"},
        "transfer_windows": {
            ("road", "rail"): [6, 7, 18, 19],
            ("rail", "road"): [11, 12, 23],
            ("road", "air"): [9, 10, 17, 18],
            ("air", "road"): [13, 14, 21, 22],
            ("road", "ship"): [6, 7, 18, 19],
            ("ship", "road"): [10, 11, 22, 23],
            ("rail", "ship"): [7, 19],
            ("ship", "rail"): [9, 21],
        },
    },
    {
        "id": "FRA",
        "name": "Frankfurt",
        "supported_modes": {"road", "rail", "air", "ship"},
        "transfer_windows": {
            ("road", "rail"): [5, 6, 17, 18],
            ("rail", "road"): [9, 10, 21],
            ("road", "air"): [6, 7, 14, 15],
            ("air", "road"): [10, 11, 18, 19],
            ("road", "ship"): [4, 5, 16, 17],
            ("ship", "road"): [10, 11, 22],
            ("rail", "ship"): [6, 18],
            ("ship", "rail"): [8, 20],
        },
    },
    {
        "id": "MUC",
        "name": "Munich",
        "supported_modes": {"road", "rail", "air"},
        "transfer_windows": {
            ("road", "rail"): [4, 5, 19, 20],
            ("rail", "road"): [9, 10, 22],
            ("road", "air"): [8, 9, 18, 19],
            ("air", "road"): [12, 13, 22],
        },
    },
]

hubs_by_id = {hub["id"]: hub for hub in hubs}
assert len(hubs_by_id) == len(hubs), "Hub-IDs have to be unique."

mode_factors = {
    "road": {"cost_per_ton_km": 1.20, "emissions_kg_per_ton_km": 0.09, "speed_kmh": 70},
    "rail": {
        "cost_per_ton_km": 0.70,
        "emissions_kg_per_ton_km": 0.025,
        "speed_kmh": 60,
    },
    "air": {"cost_per_ton_km": 3.50, "emissions_kg_per_ton_km": 0.60, "speed_kmh": 500},
    "ship": {
        "cost_per_ton_km": 0.40,
        "emissions_kg_per_ton_km": 0.015,
        "speed_kmh": 25,
    },
}

# Jede Transportkante besitzt ihren eigenen täglichen Fahrplan.
# departure_hours=[6, 18] bedeutet: Diese konkrete Verbindung fährt täglich um 06:00 und 18:00 Uhr ab.
arc_templates = [
    # BER -> HAM
    {
        "id": "BER_HAM_road",
        "from": "BER",
        "to": "HAM",
        "mode": "road",
        "dist": 290,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "BER_HAM_rail",
        "from": "BER",
        "to": "HAM",
        "mode": "rail",
        "dist": 285,
        "departure_hours": [6, 18],
    },
    {
        "id": "BER_HAM_ship",
        "from": "BER",
        "to": "HAM",
        "mode": "ship",
        "dist": 350,
        "departure_hours": [4, 16],
    },
    # HAM -> FRA
    {
        "id": "HAM_FRA_road",
        "from": "HAM",
        "to": "FRA",
        "mode": "road",
        "dist": 500,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "HAM_FRA_rail",
        "from": "HAM",
        "to": "FRA",
        "mode": "rail",
        "dist": 510,
        "departure_hours": [8, 20],
    },
    {
        "id": "HAM_FRA_ship",
        "from": "HAM",
        "to": "FRA",
        "mode": "ship",
        "dist": 620,
        "departure_hours": [8, 20],
    },
    # FRA -> MUC
    {
        "id": "FRA_MUC_road",
        "from": "FRA",
        "to": "MUC",
        "mode": "road",
        "dist": 400,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "FRA_MUC_rail",
        "from": "FRA",
        "to": "MUC",
        "mode": "rail",
        "dist": 410,
        "departure_hours": [7, 19],
    },
    {
        "id": "FRA_MUC_air",
        "from": "FRA",
        "to": "MUC",
        "mode": "air",
        "dist": 380,
        "departure_hours": [8, 16],
    },
    # BER -> FRA
    {
        "id": "BER_FRA_road",
        "from": "BER",
        "to": "FRA",
        "mode": "road",
        "dist": 550,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "BER_FRA_rail",
        "from": "BER",
        "to": "FRA",
        "mode": "rail",
        "dist": 540,
        "departure_hours": [6, 18],
    },
    {
        "id": "BER_FRA_air",
        "from": "BER",
        "to": "FRA",
        "mode": "air",
        "dist": 500,
        "departure_hours": [9, 17],
    },
    # BER -> MUC
    {
        "id": "BER_MUC_road",
        "from": "BER",
        "to": "MUC",
        "mode": "road",
        "dist": 600,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "BER_MUC_rail",
        "from": "BER",
        "to": "MUC",
        "mode": "rail",
        "dist": 585,
        "departure_hours": [6, 18],
    },
    {
        "id": "BER_MUC_air",
        "from": "BER",
        "to": "MUC",
        "mode": "air",
        "dist": 520,
        "departure_hours": [9, 17],
    },
    # HAM -> MUC
    {
        "id": "HAM_MUC_road",
        "from": "HAM",
        "to": "MUC",
        "mode": "road",
        "dist": 800,
        "departure_hours": list(range(0, 24)),
    },
    {
        "id": "HAM_MUC_rail",
        "from": "HAM",
        "to": "MUC",
        "mode": "rail",
        "dist": 780,
        "departure_hours": [8, 20],
    },
    {
        "id": "HAM_MUC_air",
        "from": "HAM",
        "to": "MUC",
        "mode": "air",
        "dist": 650,
        "departure_hours": [11, 19],
    },
]

In [18]:
time_arcs = []
shipment_weight = 1.0  # in Tonnen
max_hour = 48  # Planungshorizont
hours_per_day = 24
planning_days = range((max_hour + hours_per_day - 1) // hours_per_day)

# Datenkonsistenz: Kanten und Transfers dürfen nur vorhandene Hub-Modi referenzieren.
for template in arc_templates:
    mode = template["mode"]
    from_hub = hubs_by_id[template["from"]]
    to_hub = hubs_by_id[template["to"]]
    assert (
        mode in from_hub["supported_modes"]
    ), f"{template['id']}: {mode} ist am Start-Hub nicht verfügbar."
    assert (
        mode in to_hub["supported_modes"]
    ), f"{template['id']}: {mode} ist am Ziel-Hub nicht verfügbar."

for hub in hubs:
    for from_mode, to_mode in hub.get("transfer_windows", {}):
        assert {from_mode, to_mode}.issubset(
            hub["supported_modes"]
        ), f"{hub['id']}: Transfer {from_mode}->{to_mode} verwendet einen nicht verfügbaren Modus."

# 2.1 Transport Arcs
for template in arc_templates:
    from_hub = template["from"]
    to_hub = template["to"]
    mode = template["mode"]
    dist = template["dist"]
    duration = int(np.ceil(dist / mode_factors[mode]["speed_kmh"]))

    # Die direkt an dieser Kante angegebenen Abfahrten wiederholen sich täglich.
    for day in planning_days:
        for dep_h in template["departure_hours"]:
            start_h = day * hours_per_day + dep_h
            arrival_h = start_h + duration
            if arrival_h <= max_hour:
                time_arcs.append(
                    {
                        "service_id": template["id"],
                        "from": f"{from_hub}_{mode}_{start_h}",
                        "to": f"{to_hub}_{mode}_{arrival_h}",
                        "type": "transport",
                        "mode": mode,
                        "cost": dist
                        * mode_factors[mode]["cost_per_ton_km"]
                        * shipment_weight,
                        "emissions": dist
                        * mode_factors[mode]["emissions_kg_per_ton_km"]
                        * shipment_weight,
                        "duration": duration,
                        "departure_h": start_h,
                        "arrival_h": arrival_h,
                    }
                )

# 2.2 Waiting Arcs (Warten am Hub)
# Warten ist nur in Modi möglich, für die der Hub Infrastruktur besitzt.
for hub in hubs:
    for mode in sorted(hub["supported_modes"]):
        for t in range(max_hour):
            time_arcs.append(
                {
                    "from": f"{hub['id']}_{mode}_{t}",
                    "to": f"{hub['id']}_{mode}_{t+1}",
                    "type": "waiting",
                    "mode": mode,
                    "cost": 5.0 * shipment_weight,  # Fixe Lagerkosten pro Stunde
                    "emissions": 0.0,
                    "duration": 1,
                }
            )

# 2.3 Transfer Arcs (Umladen)
for hub in hubs:
    windows = hub.get("transfer_windows", {})
    for (m1, m2), start_times in windows.items():
        for day in planning_days:
            for st in start_times:
                t = day * hours_per_day + st
                # TODO: Model more realistic data
                if t + 2 <= max_hour:  # 2h Umladezeit angenommen
                    time_arcs.append(
                        {
                            "from": f"{hub['id']}_{m1}_{t}",
                            "to": f"{hub['id']}_{m2}_{t+2}",
                            "type": "transfer",
                            "mode": f"{m1}->{m2}",
                            "cost": 50.0 * shipment_weight,  # Fixe Handlingkosten
                            "emissions": 5.0
                            * shipment_weight,  # Emissionen beim Umschlag
                            "duration": 2,
                        }
                    )

print(f"Anzahl generierter Arcs: {len(time_arcs)}")

Anzahl generierter Arcs: 1195


### Optimierungsmodell mit PuLP

Wir definieren die Inputs und die Zielfunktion.

In [19]:
# BENUTZER-EINSTELLUNGEN
START_HUB = "BER"
END_HUB = "MUC"
START_TIME = 4
DEADLINE = 36
MAX_PRICE = 3000.0

# WEIGHTS
W_COST = 0.0
W_TIME = 1.0
W_ECO = 0.0

# Problem-Definition
prob = pulp.LpProblem("Multimodal_Routing", pulp.LpMinimize)

# Entscheidungsvariablen
arc_vars = pulp.LpVariable.dicts("Route", range(len(time_arcs)), cat=pulp.LpBinary)

# Normalisierungsfaktoren (angenommen)
MAX_EST_COST = 1500.0
MAX_EST_TIME = 48.0
MAX_EST_ECO = 500.0

# Zielfunktion
prob += pulp.lpSum(
    [
        arc_vars[i]
        * (
            W_COST * (time_arcs[i]["cost"] / MAX_EST_COST)
            + W_TIME * (time_arcs[i]["duration"] / MAX_EST_TIME)
            + W_ECO * (time_arcs[i]["emissions"] / MAX_EST_ECO)
        )
        for i in range(len(time_arcs))
    ]
)

# Nebenbedingungen

# 1. Budget Constraint
prob += (
    pulp.lpSum([arc_vars[i] * time_arcs[i]["cost"] for i in range(len(time_arcs))])
    <= MAX_PRICE
)

# 2. Flusserhaltung
nodes_in_network = set()
for a in time_arcs:
    nodes_in_network.add(a["from"])
    nodes_in_network.add(a["to"])

for node in nodes_in_network:
    hub, mode, t = node.rsplit("_", 2)
    t = int(t)

    in_flow = pulp.lpSum(
        [arc_vars[i] for i in range(len(time_arcs)) if time_arcs[i]["to"] == node]
    )
    out_flow = pulp.lpSum(
        [arc_vars[i] for i in range(len(time_arcs)) if time_arcs[i]["from"] == node]
    )

    is_start_node = hub == START_HUB and t == START_TIME
    is_end_node = hub == END_HUB and t <= DEADLINE

    # Quelle und Senke erhalten unten eigene Bedingungen
    if not is_start_node and not is_end_node:
        prob += in_flow == out_flow

# 3. Quelle und Senke
# Der Einstieg darf am Starthub in jedem dort vorhandenen Modus erfolgen
# Genau eine Kante verlässt den Starthub zum Startzeitpunkt
prob += (
    pulp.lpSum(
        [
            arc_vars[i]
            for i in range(len(time_arcs))
            if time_arcs[i]["from"].startswith(f"{START_HUB}_")
            and int(time_arcs[i]["from"].split("_")[-1]) == START_TIME
        ]
    )
    == 1
)

# Genau eine Kante erreicht den Zielhub spätestens zur Deadline.
prob += (
    pulp.lpSum(
        [
            arc_vars[i]
            for i in range(len(time_arcs))
            if time_arcs[i]["to"].startswith(f"{END_HUB}_")
            and int(time_arcs[i]["to"].split("_")[-1]) <= DEADLINE
        ]
    )
    == 1
)

## Ergebnis und Auswertung

In [20]:
status = prob.solve(pulp.PULP_CBC_CMD(msg=0))

if pulp.LpStatus[status] == "Optimal":
    print("Optimale Route gefunden.\n")
    chosen_arcs = [
        time_arcs[i] for i in range(len(time_arcs)) if pulp.value(arc_vars[i]) == 1
    ]

    # Sortieren nach Zeit
    chosen_arcs.sort(key=lambda x: int(x["from"].split("_")[-1]))

    total_cost = 0
    total_eco = 0
    last_time = 0

    for a in chosen_arcs:
        print(
            f"[{a['from']}] -> [{a['to']}] | Modus: {a['mode']:10} | Typ: {a['type']:10} | Kosten: {a['cost']:6.2f} | CO2: {a['emissions']:6.2f}"
        )
        total_cost += a["cost"]
        total_eco += a["emissions"]
        last_time = int(a["to"].split("_")[-1])

    print("\n" + "=" * 30)
    print(f"Gesamtkosten:   {total_cost:10.2f} EUR")
    print(f"Gesamt-CO2:     {total_eco:10.2f} kg")
    print(f"Ankunft (h):    {last_time:10d} (Dauer: {last_time - START_TIME}h)")
    print("=" * 30)
else:
    print(f"Keine Lösung gefunden. Status: {pulp.LpStatus[status]}")

Optimale Route gefunden.

[BER_air_4] -> [BER_air_5] | Modus: air        | Typ: waiting    | Kosten:   5.00 | CO2:   0.00
[BER_air_5] -> [BER_air_6] | Modus: air        | Typ: waiting    | Kosten:   5.00 | CO2:   0.00
[BER_air_6] -> [BER_air_7] | Modus: air        | Typ: waiting    | Kosten:   5.00 | CO2:   0.00
[BER_air_7] -> [BER_air_8] | Modus: air        | Typ: waiting    | Kosten:   5.00 | CO2:   0.00
[BER_air_8] -> [BER_air_9] | Modus: air        | Typ: waiting    | Kosten:   5.00 | CO2:   0.00
[BER_air_9] -> [MUC_air_11] | Modus: air        | Typ: transport  | Kosten: 1820.00 | CO2: 312.00

Gesamtkosten:      1845.00 EUR
Gesamt-CO2:         312.00 kg
Ankunft (h):            11 (Dauer: 7h)
